# 1. Reading `nuppn` output with `NuGridJl`

This walks through the base reader layer: `PPNRun`, the lazy handle over a
single-zone `nuppn` run directory, and the typed values it hands back —
`Abundances`, fluxes, the `x-time.dat` time series, the reaction `Network`,
the isotope database, and the parsed input decks.

We use the fixture data bundled with the package (`test/data/nuppn_data/`) so
this notebook runs standalone, without needing a real NuPPN install.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using NuGridJl
using DataFrames

  Activating 

project at `~/Documents/NovaNucleosynthesis/NuGrid-Tools/NuGridJl`


## Loading a run

`PPNRun(dir)` doesn't read anything up front except the list of available
cycles (by scanning for `iso_massfNNNNN.DAT` files) — everything else is
read lazily and cached the first time you ask for it.

In [2]:
DATA = joinpath(@__DIR__, "..", "test", "data", "nuppn_data")
run = PPNRun(DATA)

PPNRun("/home/sgervais/Documents/NovaNucleosynthesis/NuGrid-Tools/NuGridJl/demos/../test/data/nuppn_data", 42 cycles 0–804)

In [3]:
first(run.cycles), last(run.cycles), length(run.cycles)

(0, 804, 42)

## Abundances

`abundances(run, cycle)` takes a cycle number, or `:initial` / `:final` /
`:decay`. Index the result by isotope (an `Isotope`, or a name string like
`"He-4"`) to get a mass fraction.

In [4]:
ab_initial = abundances(run, :initial)
ab_final = abundances(run, :final)
(initial = ab_initial["He-4"], final = ab_final["He-4"])

(initial = 0.136749, final = 0.315401)

In [5]:
# Every Abundances converts to a tidy DataFrame for further analysis.
sort(DataFrame(ab_final), :X; rev = true)[1:10, :]

Row,Z,A,N,isomer,isotope,X
,Int64,Int64,Int64,Int64,String,Float64
1,7,14,7,0,N-14,0.364434
2,2,4,2,0,He-4,0.315401
3,1,1,0,0,p,0.176478
4,7,13,6,0,N-13,0.0432751
5,6,12,6,0,C-12,0.0273562
6,6,13,7,0,C-13,0.0239121
7,8,16,8,0,O-16,0.0230909
8,14,28,14,0,Si-28,0.00883811
9,10,20,10,0,Ne-20,0.00546418


## Fluxes

`fluxes(run, cycle)` reads `flux_NNNNN.DAT` — one row per reaction, with the
reactant/product isotopes and the flux (dY/dt) carried at that cycle.

In [6]:
fx = fluxes(run, :final)
sort(fx, :flux; rev = true)[1:8, [:index, :reactant, :product, :flux]]

Row,index,reactant,product,flux
,Int64,Isotope,Isotope,Float64
1,161,C-13,N-14,4.29619e-6
2,181,N-13,C-13,3.86147e-6
3,154,C-12,N-13,1.41068e-6
4,227,O-15,N-15,1.33556e-6
5,201,N-15,C-12,1.33411e-6
6,17,N-14,O-15,1.19944e-6
7,245,O-17,N-14,4.7925e-8
8,215,O-14,N-14,1.79267e-8


## Time series (`x-time.dat`)

`xtime(run)` reads the whole-run time series once and caches it.
`series(xt, iso; x = :time)` pulls out one isotope's mass fraction against
time (or `:t9`/`:rho`).

In [7]:
xt = xtime(run)
t, X = series(xt, "He-4"; x = :time)
(n_points = length(t), t_first = t[1], t_last = t[end], X_last = X[end])

(n_points = 805, t_first = 0.0, t_last = 0.000128265, X_last = 0.315401)

## The reaction network

`network(run)` parses `networksetup.txt`: every tracked isotope, and every
reaction row (active or not), with its source label, printed rate, applied
multiplier and REACLIB chapter.

In [8]:
net = network(run)
(n_isotopes = length(net.isotopes), n_reactions = length(net.reactions),
 n_active = count(r -> r.active, net.reactions))

(n_isotopes = 1093, n_reactions = 14007, n_active = 13858)

In [9]:
he4_reactions = reactions_for_isotope(net, Isotope(2, 4, 0))
describe_rate.(he4_reactions[1:5])

5-element Vector{String}:
 "3He(v,v)4He        source=VITAL  rate=1.9820e-15  x1"
 "4He(v,v)7Be        source=VITAL  rate=1.1560e-20  x1"
 "7Li(p,g)4He        source=VITAL  rate=4.6590e-09  x1"
 "8B(v,v)4He         source=VITAL  rate=8.9600e-01  x1"
 "7Be(a,g)11C        source=VITAL  rate=0.0000e+00  x1"

## Isotope database and input decks

`isotopedatabase(run)` reads `isotopedatabase.txt` (every isotope NuGrid
knows how to add to a network, and whether it's currently active).
`inputs(run)` gives you the three parsed `ppn_*.input` namelists.

In [10]:
db = isotopedatabase(run)
length(db), count(e -> e.active, db)

(1107, 1085)

In [11]:
in_ = inputs(run)
in_.physics[:NVCP], in_.physics[:NRCP], in_.physics[:INDEX_REACLIB]

(55, 117, 2)

## Next

[`02_abundance_and_flux_charts.ipynb`](02_abundance_and_flux_charts.ipynb) —
turn `ab_final`, `fx` and `net` into the nuclear-chart plots.